[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_04/SFM_ch4_dependence/SFM_ch4_dependence.ipynb)

# SFM_ch4_dependence

Joint distributions and dependence: scatter plots of daily returns (S&P 500 vs DAX and vs BET, common trading days), covariance, correlation and the volatility of a 50/50 portfolio; uncorrelated but dependent variables (X and X^2; returns and squared returns of four markets); conditional mean and conditional standard deviation of today's return given the quintile of yesterday's absolute return, with the law of total variance.

*Statistics of Financial Markets (SFM), Chapter 4: Probability. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_4'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
START = '2000-01-01'

ASSETS = ['sp500', 'dax', 'bet', 'btc']

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F'}

SHORT = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin'}

PAL = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F', 'purple': '#8E44AD', 'teal': '#17A2B8', 'orange': '#E67E22', 'crimson': '#DC3545'}

RANDU_A, RANDU_M = 65539, 2147483648

def returns(name, start=START, end=None):
    """Daily log returns in %, from `start` or from the first day of the series if that is later."""
    s0 = max(start, SERIES[name][4])
    return log_returns(name, s0) if end is None else log_returns(name, s0, end)


def joint_returns(a, b, start=START):
    """Daily log returns (%) of two series on their common trading days: join the prices first, then difference."""
    p = load_panel([a, b], start=start).dropna()
    return (100 * np.log(p).diff()).dropna()


def acf(x, nlags=20):
    """Sample autocorrelation function rho(1), ..., rho(nlags)."""
    x = np.asarray(x, dtype=float) - np.mean(x)
    d = np.sum(x * x)
    return np.array([np.sum(x[h:] * x[:-h]) / d for h in range(1, nlags + 1)])


def max_drawdown(prices):
    """Maximum drawdown of a price path: the largest fall from a running peak, min_t (P_t / max_{s<=t} P_s - 1)."""
    p = np.asarray(prices, dtype=float)
    return float(np.min(p / np.maximum.accumulate(p) - 1))


def annual_days(r):
    """A, the number of periods per year used to annualise: 252 trading days for exchange-traded series,
    365 for crypto assets, which trade every calendar day."""
    return 365 if periods_per_year(r) > 300 else 252


def gbm_params(r, ppy=None):
    """Calibrate GBM to daily log returns r (in %): annual log drift mu_log, volatility sigma, arithmetic drift
    mu = mu_log + sigma^2/2 (so that E[S_t] = S_0 exp(mu t) and median S_0 exp(mu_log t))."""
    r = pd.Series(r).dropna() / 100
    ppy = ppy or annual_days(r)
    m, s = r.mean() * ppy, r.std() * np.sqrt(ppy)
    return {'mu_log': m, 'sigma': s, 'mu': m + s ** 2 / 2, 'ppy': ppy, 'n': len(r)}


def simulate_gbm(S0, mu, sigma, T, steps, n_paths, seed=0):
    """Exact simulation of GBM: S_{t+dt} = S_t exp((mu - sigma^2/2) dt + sigma sqrt(dt) Z), Z ~ N(0, 1).
    Returns an array (n_paths, steps + 1). Port of SFEsimGBM."""
    rng = np.random.default_rng(seed)
    dt = T / steps
    z = rng.standard_normal((n_paths, steps))
    logp = np.cumsum((mu - sigma ** 2 / 2) * dt + sigma * np.sqrt(dt) * z, axis=1)
    return S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), logp]))


def lcg(a, c, M, seed, n):
    """Linear congruential generator x_{k+1} = (a x_k + c) mod M; returns the integers and the uniforms x/M.
    Port of SFErangen1/SFErangen2."""
    x = np.empty(n, dtype=np.int64)
    x[0] = seed
    for k in range(1, n):
        x[k] = (a * int(x[k - 1]) + c) % M
    return x, x / M


def lcg_period(a, c, M, seed):
    """Length of the cycle of an LCG started at `seed`."""
    seen, x, k = {}, seed, 0
    while x not in seen:
        seen[x] = k
        x = (a * x + c) % M
        k += 1
    return k - seen[x]


def randu(n, seed=1):
    """The RANDU generator (multiplier 65539, modulus 2^31, odd seed). Port of SFErandu."""
    return lcg(RANDU_A, 0, RANDU_M, seed, n)


def binomial_crr(S0, mu, sigma, T, n):
    """Cox-Ross-Rubinstein binomial process: n steps of length dt = T/n, up factor u = exp(sigma sqrt(dt)),
    down factor d = 1/u, probability of an up move p = (exp(mu dt) - d) / (u - d), so that E[S_{t+dt}] = S_t exp(mu dt).
    Returns the terminal values S_0 u^k d^(n-k), their probabilities and the parameters. Port of SFEBinomp."""
    dt = T / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(mu * dt) - d) / (u - d)
    k = np.arange(n + 1)
    return {'S': S0 * u ** k * d ** (n - k), 'prob': stats.binom.pmf(k, n, p), 'u': u, 'd': d, 'p': p, 'dt': dt}


def ar1(phi, eps, x0=0.0):
    """AR(1) process X_t = phi X_{t-1} + eps_t driven by the shocks eps (phi = 1: random walk)."""
    x = np.empty(len(eps))
    prev = x0
    for t, e in enumerate(eps):
        prev = phi * prev + e
        x[t] = prev
    return x


def pair_stats(a, b, start=START):
    """Means, standard deviations, covariance and correlation of two return series on common days, and the
    standard deviation of the 50/50 portfolio sd_p = sqrt(w1^2 s1^2 + w2^2 s2^2 + 2 w1 w2 cov)."""
    R = joint_returns(a, b, start)
    x, y = R[a], R[b]
    cov = float(np.cov(x, y)[0, 1])
    s1, s2 = float(x.std()), float(y.std())
    sp = np.sqrt(0.25 * s1 ** 2 + 0.25 * s2 ** 2 + 0.5 * cov)
    return {'n': len(R), 'm1': float(x.mean()), 'm2': float(y.mean()), 's1': s1, 's2': s2, 'cov': cov,
            'corr': float(np.corrcoef(x, y)[0, 1]), 'sd_p': float(sp), 'sd_avg': (s1 + s2) / 2,
            'var_sum': s1 ** 2 / 4 + s2 ** 2 / 4, 'cov_term': 0.5 * cov}


def fig_joint(pairs=(('sp500', 'dax'), ('sp500', 'bet')), save=True):
    """Scatter plots of daily returns of two markets on common days (joint distribution) with the correlation."""
    fig, axes = plt.subplots(1, len(pairs), figsize=(10.5, 4.0))
    out = {}
    for ax, (a, b) in zip(axes, pairs):
        R = joint_returns(a, b)
        o = pair_stats(a, b)
        ax.scatter(R[a].clip(-12, 12), R[b].clip(-12, 12), s=3, alpha=0.35, color=COLORS[b],
                   label=f'{SHORT[a]} vs {SHORT[b]}: correlation {o["corr"]:.2f}')
        ax.axhline(0, color=st.DarkText, lw=0.5)
        ax.axvline(0, color=st.DarkText, lw=0.5)
        ax.set_xlabel(f'{SHORT[a]} daily log return (%)')
        ax.set_ylabel(f'{SHORT[b]} daily log return (%)')
        ax.set_xlim(-12, 12)
        ax.set_ylim(-12, 12)
        out[f'{a}_{b}'] = o
    st.fig_legend_bottom(fig, ncol=2, y=0.02)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch4_joint')
    return out


def lag_dependence(r):
    """Corr(r_t, r_{t-1}) and Corr(r_t^2, r_{t-1}^2) with the i.i.d. band 1.96/sqrt(n)."""
    r = pd.Series(r).dropna().values
    return {'n': len(r), 'rho_r': float(np.corrcoef(r[1:], r[:-1])[0, 1]),
            'rho_r2': float(np.corrcoef(r[1:] ** 2, r[:-1] ** 2)[0, 1]),
            'rho_abs': float(np.corrcoef(np.abs(r[1:]), np.abs(r[:-1]))[0, 1]), 'band': 1.96 / np.sqrt(len(r))}


def fig_uncorrelated_dependent(names=ASSETS, seed=4, save=True):
    """Left: X ~ N(0,1) and Y = X^2 are uncorrelated but dependent. Right: for real returns, Corr(r_t, r_{t-1}) is
    close to zero while Corr(r_t^2, r_{t-1}^2) is clearly positive."""
    rng = np.random.default_rng(seed)
    X = rng.standard_normal(3000)
    Y = X ** 2
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), gridspec_kw={'width_ratios': [1, 1.3]})
    ax = axes[0]
    ax.scatter(X, Y, s=4, alpha=0.4, color=PAL['purple'], label=f'simulated: Y = X^2, Corr(X, Y) = {np.corrcoef(X, Y)[0, 1]:.3f}')
    ax.set_xlabel('X ~ N(0, 1)')
    ax.set_ylabel('Y = X^2')
    ax = axes[1]
    out = {'xy_corr': float(np.corrcoef(X, Y)[0, 1])}
    idx = np.arange(len(names))
    rr = [lag_dependence(returns(k)) for k in names]
    ax.bar(idx - 0.18, [o['rho_r'] for o in rr], width=0.36, color=PAL['blue'], label='Corr(r_t, r_{t-1})')
    ax.bar(idx + 0.18, [o['rho_r2'] for o in rr], width=0.36, color=PAL['red'], label='Corr(r_t^2, r_{t-1}^2)')
    for i, o in enumerate(rr):
        ax.plot([i - 0.42, i + 0.42], [o['band']] * 2, color=PAL['green'], lw=1.2, ls='--',
                label='i.i.d. band +/- 1.96/sqrt(n)' if i == 0 else '_')
        ax.plot([i - 0.42, i + 0.42], [-o['band']] * 2, color=PAL['green'], lw=1.2, ls='--', label='_')
    ax.axhline(0, color=st.DarkText, lw=0.6)
    ax.set_xticks(idx)
    ax.set_xticklabels([SHORT[k] for k in names])
    ax.set_ylabel('lag-1 correlation')
    for k, o in zip(names, rr):
        out[k] = o
    st.fig_legend_bottom(fig, ncol=2, y=0.02)
    fig.tight_layout(rect=(0, 0.14, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch4_uncorrelated_dependent')
    return out


def conditional_moments(r, q=5):
    """Conditional mean and standard deviation of r_t given the quintile of |r_{t-1}|; the law of total variance
    Var(r) = E[Var(r | Q)] + Var(E[r | Q])."""
    r = pd.Series(r).dropna()
    prev = r.shift(1).abs()
    df = pd.DataFrame({'r': r, 'prev': prev}).dropna()
    df['Q'] = pd.qcut(df['prev'], q, labels=False) + 1
    g = df.groupby('Q')['r']
    mean, sd, cnt = g.mean(), g.std(ddof=0), g.count()
    w = cnt / cnt.sum()
    within = float((w * sd ** 2).sum())
    between = float((w * (mean - df['r'].mean()) ** 2).sum())
    edges = df.groupby('Q')['prev'].agg(['min', 'max'])
    return {'mean': mean.values, 'sd': sd.values, 'n': cnt.values, 'within': within, 'between': between,
            'total': float(df['r'].var(ddof=0)), 'ratio': float(sd.iloc[-1] / sd.iloc[0]),
            'sd_all': float(df['r'].std(ddof=0)), 'lo': edges['min'].values, 'hi': edges['max'].values}


def fig_conditional(names=('sp500', 'btc'), save=True):
    """Conditional mean and conditional standard deviation of today's return given the quintile of yesterday's
    absolute return."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.7))
    out = {}
    idx = np.arange(1, 6)
    for j, k in enumerate(names):
        c = conditional_moments(returns(k))
        out[k] = c
        off = -0.18 if j == 0 else 0.18
        axes[0].bar(idx + off, c['mean'], width=0.36, color=COLORS[k], label=SHORT[k])
        axes[1].bar(idx + off, c['sd'], width=0.36, color=COLORS[k], label='_')
        axes[1].plot([0.6, 5.4], [c['sd_all']] * 2, color=COLORS[k], lw=1.1, ls='--',
                     label=f'{SHORT[k]}: unconditional sd')
    axes[0].axhline(0, color=st.DarkText, lw=0.6)
    axes[0].set_ylabel('E[r_t | quintile] (%)')
    axes[1].set_ylabel('sd(r_t | quintile) (%)')
    for ax in axes:
        ax.set_xticks(idx)
        ax.set_xticklabels(['Q1\ncalm', 'Q2', 'Q3', 'Q4', 'Q5\nturbulent'])
        ax.set_xlabel('quintile of |r_{t-1}|')
    st.fig_legend_bottom(fig, ncol=4, y=0.02)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch4_conditional')
    return out

## Run

In [ ]:
print(fig_joint())
print(fig_uncorrelated_dependent())
res = fig_conditional()
print({k: {'sd': v['sd'].round(3), 'ratio': round(v['ratio'], 2)} for k, v in res.items()})

![sfm_ch4_joint](./sfm_ch4_joint.png)

Output: `sfm_ch4_joint.pdf`

![sfm_ch4_uncorrelated_dependent](./sfm_ch4_uncorrelated_dependent.png)

Output: `sfm_ch4_uncorrelated_dependent.pdf`

![sfm_ch4_conditional](./sfm_ch4_conditional.png)

Output: `sfm_ch4_conditional.pdf`